# 05 — Model Explainability (SHAP)

Apply SHAP to the best model to understand which features drive predictions.

**Input:** `data/processed/playoff_features.parquet`, best model from `models/`  
**Output:** SHAP beeswarm plot, dependence plots, example game explanation

**Key questions:**
- Which features matter most globally?
- How does net rating differential interact with home court?
- Can we explain a specific upset prediction?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap

sys.path.insert(0, str(Path().resolve().parent))
from src.models import FEATURE_COLS, TARGET_COL, load_model, train_test_split_by_season

In [ ]:
# CONFIG
PROCESSED_DIR = Path().resolve().parent / "data" / "processed"
FEATURES_PATH = PROCESSED_DIR / "playoff_features.parquet"
TEST_SEASONS = ["2022-23", "2023-24"]
BEST_MODEL_NAME = "xgboost"  # update if a different model won in notebook 04

## 1. Load Data + Model

In [ ]:
df = pd.read_parquet(FEATURES_PATH)
_, test_df = train_test_split_by_season(df, TEST_SEASONS)
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

model = load_model(BEST_MODEL_NAME)
print(f"Loaded model: {BEST_MODEL_NAME}")
print(f"Test set: {len(X_test)} games")

## 2. Compute SHAP Values

In [ ]:
# For tree models (RF, XGBoost) use TreeExplainer — fast and exact
# For Logistic Regression use LinearExplainer
clf = model.named_steps["clf"]
X_test_scaled = model.named_steps["scaler"].transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=FEATURE_COLS)

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_scaled_df)
print(f"SHAP values shape: {np.array(shap_values).shape}")

## 3. Global Feature Importance — Beeswarm Plot

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_scaled_df,
    plot_type="beeswarm",
    max_display=15,
    show=True,
)

## 4. Bar Chart — Mean Absolute SHAP

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_scaled_df,
    plot_type="bar",
    max_display=15,
    show=True,
)

## 5. Dependence Plot — Net Rating Differential

In [ ]:
# How does net_rtg difference affect prediction probability?
# Color by home_rest_days to show interaction
shap.dependence_plot(
    "ortg_diff",
    shap_values,
    X_test_scaled_df,
    interaction_index="home_rest_days",
    show=True,
)

## 6. Single Game Explanation

In [ ]:
# Pick an interesting game — e.g. a model-predicted upset that actually happened
# TODO: identify game index where y_pred=1, y_test=1 but home team was the lower seed
game_idx = 0  # replace with interesting game index

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[game_idx],
        base_values=explainer.expected_value,
        data=X_test_scaled_df.iloc[game_idx],
        feature_names=FEATURE_COLS,
    )
)